# 07 — Series clustering by usage profile

Groups the series into homogeneous profiles via K-Means on aggregated features.

See the rationale and feature description in [docs/06_clustering_recommendations.md](../docs/06_clustering_recommendations.md).

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_parquet, filter_period, build_series
from src.plotting import plot_scatter_clusters, plot_heatmap

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'clustering'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Loading the series

In [ ]:
df = load_parquet(REPO_ROOT / 'data' / 'anonymized_series.parquet')
df = filter_period(df, 2020, 2024)
series = build_series(df, min_months=24)

## 3. Per-series feature computation

Six features described in the docs. The TabPFN-derived ones can be added if available.

In [ ]:
def series_features(values: np.ndarray) -> dict:
    v = values.astype(np.float64)
    n = len(v)
    mean = v.mean()
    std = v.std()
    slope = np.polyfit(np.arange(n), v, 1)[0]
    cv = std / mean if mean > 0 else 0.0
    monthly_means = np.array([v[i::12].mean() for i in range(12)])
    seasonality = np.var(monthly_means) / np.var(v) if np.var(v) > 0 else 0.0
    zero_prop = float((v == 0).sum() / n)
    if n >= 36:
        recent_ratio = v[-12:].mean() / max(v[:24].mean(), 1e-6)
        recent_ratio = min(recent_ratio, 3.0)
    else:
        recent_ratio = 1.0
    return {
        'mean_volume': mean,
        'trend_slope': slope,
        'cv': cv,
        'seasonality_strength': seasonality,
        'zero_prop': zero_prop,
        'recent_ratio': recent_ratio,
    }

rows = [{'series_id': sid, **series_features(s.values)} for sid, s in series.items()]
features_df = pd.DataFrame(rows).set_index('series_id')
features_df.to_csv(OUTPUT_DIR / 'clustering_features_raw.csv')
features_df.describe()

## 4. Drop zero-variance features

In [ ]:
variances = features_df.var()
valid_cols = variances[variances > 1e-12].index.tolist()
X = features_df[valid_cols].values
X_scaled = StandardScaler().fit_transform(X)
print(f'Features used: {valid_cols}')

## 5. K selection by silhouette

In [ ]:
silhouettes = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    silhouettes[k] = silhouette_score(X_scaled, km.labels_)
pd.Series(silhouettes, name='silhouette').to_csv(OUTPUT_DIR / 'silhouettes.csv')
silhouettes

## 6. Final K-Means

K is fixed by silhouette analysis + interpretability. Adjust `K_FINAL` according to the previous result.

In [ ]:
K_FINAL = 4   # adjust based on silhouettes
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit(X_scaled)
labels = kmeans.labels_
features_df['cluster_id'] = labels

## 7. Labeling by centroid interpretation

In [ ]:
centroids = pd.DataFrame(
    StandardScaler().fit(X).inverse_transform(kmeans.cluster_centers_),
    columns=valid_cols,
)
centroids.index.name = 'cluster_id'
centroids.to_csv(OUTPUT_DIR / 'cluster_centroids.csv')

def cluster_label(row):
    if row['recent_ratio'] < 0.9 and row['trend_slope'] < 0:
        return 'D_decline'
    if row['recent_ratio'] > 1.3 and row['trend_slope'] > 0:
        return 'B_growth'
    if row.get('zero_prop', 0) > 0.2 or row['cv'] > 1.5:
        return 'C_intermittent'
    return 'A_consolidated'

label_map = centroids.apply(cluster_label, axis=1).to_dict()
features_df['cluster_label'] = features_df['cluster_id'].map(label_map)
features_df.to_csv(OUTPUT_DIR / 'series_profiles.csv')
features_df['cluster_label'].value_counts()

## 8. Visualizations

In [ ]:
pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plot_scatter_clusters(
    pca, labels, 'Series clusters (PCA 2D)',
    OUTPUT_DIR / 'scatter_clusters.png',
)

plot_heatmap(
    centroids.round(2),
    'Cluster centroids (original space)',
    OUTPUT_DIR / 'heatmap_centroids.png',
    cmap='RdYlBu_r',
)

## 9. Cross with the model metrics (from notebook 06)

In [ ]:
metrics_multi = pd.read_csv(REPO_ROOT / 'outputs' / 'comparison' / 'multimodel_comparison_metrics.csv')
cross = metrics_multi.merge(
    features_df.reset_index()[['series_id', 'cluster_label']],
    on='series_id',
)
table = cross.groupby(['cluster_label', 'model'])['MASE'].median().unstack()
table.to_csv(OUTPUT_DIR / 'cluster_vs_model_mase.csv')
table.round(3)